# METAGENE-1 Shotgun Metagenomic Analysis — Gujarat India Wastewater
**95 samples · 4 cities · 4 seasons**

## v3 — Disconnect-proof
- Cell 3 auto-restores results from HuggingFace on every restart
- Cell 8 auto-saves after every single sample
- Colab disconnections are completely harmless

## Before running
1. Runtime → Change runtime type → **A100 GPU**
2. Paste your HF token in Cell 2
3. Runtime → Run all

In [ ]:
# Cell 1: Set your HuggingFace token here (only place you need to edit)
HF_TOKEN = 'PASTE_YOUR_NEW_HF_TOKEN_HERE'  # get from huggingface.co/settings/tokens

import os
os.environ['HF_TOKEN'] = HF_TOKEN
print('Token set.')

In [ ]:
# Cell 2: Install dependencies
import subprocess, sys
pkgs = ['transformers>=4.40', 'accelerate', 'safetensors', 'sentencepiece',
        'umap-learn', 'seaborn', 'huggingface_hub', 'scipy', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Dependencies installed.')

In [ ]:
# Cell 3: GPU check
import torch
assert torch.cuda.is_available(), 'No GPU! Change runtime to A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')
if   'A100' in gpu_name: BATCH_SIZE = 64
elif 'V100' in gpu_name or 'A10' in gpu_name: BATCH_SIZE = 32
else: BATCH_SIZE = 16
print(f'Batch size: {BATCH_SIZE}')

In [ ]:
# Cell 4: Download data + AUTO-RESTORE saved results from HuggingFace
# Every restart: data re-downloaded, results re-restored. Disconnections are harmless.
import os, shutil
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

HF_REPO    = 'saurabhthakar3/gujarat-wastewater-shotgun'
DATA_ROOT  = Path('/content/gujarat_shotgun')
RESULTS_DIR = Path('/content/results')
DATA_ROOT.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ── Step 1: Download FASTA + metadata ────────────────────────────────────────
print('Step 1: Downloading FASTA data from HuggingFace...')
snapshot_download(
    repo_id=HF_REPO, repo_type='dataset',
    local_dir=str(DATA_ROOT), token=HF_TOKEN,
    ignore_patterns=['results/*'],   # skip results here, restore separately below
)
print('Data downloaded.')

# ── Step 2: Restore saved results ────────────────────────────────────────────
print('\nStep 2: Checking for saved results on HuggingFace...')
api = HfApi()
try:
    all_files    = list(api.list_repo_files(repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN))
    result_files = [f for f in all_files if f.startswith('results/')]
    if result_files:
        print(f'Found {len(result_files)} saved result files — restoring...')
        snapshot_download(
            repo_id=HF_REPO, repo_type='dataset',
            local_dir=str(DATA_ROOT), token=HF_TOKEN,
            allow_patterns=['results/*'],
        )
        # Copy from DATA_ROOT/results/ → /content/results/
        src = DATA_ROOT / 'results'
        if src.exists():
            for f in src.glob('*'):
                shutil.copy2(f, RESULTS_DIR / f.name)
        n_restored = len(list(RESULTS_DIR.glob('*_anomaly.json')))
        print(f'Restored {n_restored} sample results — Cell 8 will skip these.')
    else:
        print('No saved results found — Cell 8 will compute all 95 samples.')
except Exception as e:
    print(f'Could not restore results: {e} — will compute from scratch.')

# ── Step 3: Set paths ────────────────────────────────────────────────────────
FASTA_DIR        = DATA_ROOT / 'phase3_fasta'
METADATA_CSV     = DATA_ROOT / 'sample_metadata_complete.csv'
KRAKEN_DIR       = DATA_ROOT / 'kraken2_results'
KRAKEN_VIRAL_DIR = DATA_ROOT / 'kraken2_viral_results'
MODEL_CACHE      = Path('/content/model_cache')
MODEL_CACHE.mkdir(exist_ok=True)

fastas = sorted(FASTA_DIR.glob('*.fasta.gz'))
print(f'\nFASTA files : {len(fastas)}')
print(f'Metadata    : {METADATA_CSV.exists()}')
assert len(fastas) > 0, 'No FASTA files found'

In [ ]:
# Cell 5: Configuration
import pandas as pd

MODEL_ID           = 'metagene-ai/METAGENE-1'
N_READS_PER_SAMPLE = 5_000     # matches Liu et al. 2025
ANOMALY_THRESHOLD  = 3.0
MAX_SEQ_LEN        = 512
CHUNK_SIZE         = 50_000
SEED               = 42

CITY_COLORS    = {'Ahmedabad':'#1f77b4','Surat':'#ff7f0e','Rajkot':'#2ca02c','Vadodara':'#d62728'}
SEASON_MARKERS = {'Summer':'o','Monsoon':'s','PreWinter':'^','Winter':'D'}
SEASON_ORDER   = ['Summer','Monsoon','PreWinter','Winter']

# Load metadata and fix sample_id to match FASTA filenames (strip _R1)
meta_raw = pd.read_csv(METADATA_CSV)
print('Metadata sample_id example :', meta_raw['sample_id'].iloc[0])
print('FASTA filename example      :', fastas[0].stem.replace('.fasta',''))

# Build lookup: FASTA stem (e.g. AAMO_R1) → metadata row
# Strip _R1 from FASTA stem to match metadata sample_id
meta = meta_raw.copy()
meta['fasta_id'] = meta['sample_id']   # will be used as merge key

print(f'\nConfig OK | {len(fastas)} samples | {N_READS_PER_SAMPLE:,} reads/sample')

In [ ]:
# Cell 6: Load METAGENE-1
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda')
print(f'Loading {MODEL_ID}...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float16, device_map='cuda')
model.eval()
print(f'Loaded in {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 7: Helper functions
import gzip, random, json
import numpy as np
from huggingface_hub import HfApi
random.seed(SEED); np.random.seed(SEED)

def stream_fasta(path, n_reads=None, seed=SEED):
    path = str(path)
    opener = gzip.open if path.endswith('.gz') else open
    if n_reads is not None:
        rng = random.Random(seed)
        res_h, res_s, count = [], [], 0
        with opener(path, 'rt') as f:
            hdr, seq = None, []
            for line in f:
                line = line.rstrip()
                if line.startswith('>'):
                    if hdr:
                        count += 1; s = ''.join(seq)
                        if len(res_h) < n_reads: res_h.append(hdr); res_s.append(s)
                        else:
                            j = rng.randint(0, count-1)
                            if j < n_reads: res_h[j]=hdr; res_s[j]=s
                    hdr, seq = line, []
                else: seq.append(line)
            if hdr:
                count += 1; s = ''.join(seq)
                if len(res_h) < n_reads: res_h.append(hdr); res_s.append(s)
                else:
                    j = rng.randint(0, count-1)
                    if j < n_reads: res_h[j]=hdr; res_s[j]=s
        for i in range(0, len(res_h), CHUNK_SIZE):
            yield res_h[i:i+CHUNK_SIZE], res_s[i:i+CHUNK_SIZE]
    else:
        with opener(path, 'rt') as f:
            hdrs, seqs, hdr, seq = [], [], None, []
            for line in f:
                line = line.rstrip()
                if line.startswith('>'):
                    if hdr:
                        hdrs.append(hdr); seqs.append(''.join(seq))
                        if len(seqs)>=CHUNK_SIZE: yield hdrs,seqs; hdrs,seqs=[],[]
                    hdr, seq = line, []
                else: seq.append(line)
            if hdr: hdrs.append(hdr); seqs.append(''.join(seq))
            if seqs: yield hdrs, seqs

@torch.no_grad()
def process_batch(sequences):
    all_embs, all_scores = [], []
    for i in range(0, len(sequences), BATCH_SIZE):
        batch   = sequences[i:i+BATCH_SIZE]
        inputs  = tokenizer(batch, return_tensors='pt', padding=True,
                            truncation=True, max_length=MAX_SEQ_LEN,
                            add_special_tokens=False).to(device)
        outputs = model(**inputs, output_hidden_states=True, labels=inputs['input_ids'])
        hidden  = outputs.hidden_states[-1].float()
        mask    = inputs['attention_mask'].unsqueeze(-1).float()
        emb     = (hidden * mask).sum(1) / mask.sum(1)
        all_embs.append(emb.cpu().numpy())
        sl = outputs.logits[...,:-1,:].contiguous().float()
        tl = inputs['input_ids'][...,1:].contiguous()
        sm = inputs['attention_mask'][...,1:].contiguous().float()
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        tok_loss = loss_fct(sl.view(-1,sl.size(-1)),tl.view(-1)).view(tl.size())
        norm_loss = (tok_loss*sm).sum(1)/sm.sum(1).clamp(min=1)
        all_scores.append(norm_loss.cpu().numpy())
        del outputs, inputs
    return np.vstack(all_embs), np.concatenate(all_scores)

def summarise_scores(scores):
    hc, he = np.histogram(scores, bins=100, range=(0.0, 8.0))
    return {'n_reads':int(len(scores)),'mean_loss':float(scores.mean()),
            'median_loss':float(np.median(scores)),'std_loss':float(scores.std()),
            'p25_loss':float(np.percentile(scores,25)),'p75_loss':float(np.percentile(scores,75)),
            'p90_loss':float(np.percentile(scores,90)),'p95_loss':float(np.percentile(scores,95)),
            'pct_above_2':float((scores>2.0).mean()*100),
            'pct_above_3':float((scores>ANOMALY_THRESHOLD).mean()*100),
            'pct_above_4':float((scores>4.0).mean()*100),
            'hist_counts':hc.tolist(),'hist_edges':he.tolist()}

def save_sample_to_hf(sample_id):
    """Upload a single sample's results to HuggingFace immediately after processing."""
    api = HfApi()
    for suffix in ['_anomaly.json', '_embedding.npy', '_scores.npy']:
        fpath = RESULTS_DIR / f'{sample_id}{suffix}'
        if fpath.exists():
            try:
                api.upload_file(
                    path_or_fileobj=str(fpath),
                    path_in_repo=f'results/{fpath.name}',
                    repo_id=HF_REPO,
                    repo_type='dataset',
                    token=HF_TOKEN,
                )
            except Exception as e:
                print(f'  [HF save warning: {e}]')

print('Helpers ready.')
print('Paper reference: WW in-dist ~1.24 | human genome ~5.22 | random ~5.83')

In [ ]:
# Cell 8: Main loop — auto-saves each sample to HuggingFace immediately
# Safe to interrupt and resume — completed samples are permanently saved
import time, json
import numpy as np

fastas = sorted(FASTA_DIR.glob('*.fasta.gz'))
done, skipped = 0, 0
t_session = time.time()

n_already = len(list(RESULTS_DIR.glob('*_anomaly.json')))
print(f'Processing {len(fastas)} samples | {N_READS_PER_SAMPLE:,} reads/sample')
print(f'{n_already} already done (restored from HuggingFace) — will skip these')
print('─'*70)

for fasta in fastas:
    sid       = fasta.stem.replace('.fasta', '')
    out_emb   = RESULTS_DIR / f'{sid}_embedding.npy'
    out_stats = RESULTS_DIR / f'{sid}_anomaly.json'
    out_sc    = RESULTS_DIR / f'{sid}_scores.npy'

    if out_emb.exists() and out_stats.exists():
        skipped += 1
        continue

    print(f'\n[{done+skipped+1}/{len(fastas)}] {sid}')
    t0 = time.time()
    emb_sum, emb_count, sc_chunks, n_proc = None, 0, [], 0

    for _, chunk_seqs in stream_fasta(fasta, n_reads=N_READS_PER_SAMPLE):
        embs, scores = process_batch(chunk_seqs)
        emb_sum = embs.sum(0).astype(np.float64) if emb_sum is None else emb_sum+embs.sum(0)
        emb_count += len(embs); n_proc += len(chunk_seqs); sc_chunks.append(scores)
        rate = n_proc/(time.time()-t0)
        eta  = ((N_READS_PER_SAMPLE or 2_000_000)-n_proc)/max(rate,1)
        print(f'  {n_proc:>6,} reads | {rate:>5.0f} seq/s | ETA {eta/60:.1f} min', end='\r')

    print()
    np.save(out_emb, (emb_sum/emb_count).astype(np.float32))
    all_sc = np.concatenate(sc_chunks)
    np.save(out_sc, all_sc)
    st = summarise_scores(all_sc)
    st.update({'sample_id':sid,'elapsed_min':round((time.time()-t0)/60,2)})
    json.dump(st, open(out_stats,'w'), indent=2)
    done += 1
    print(f'  mean_loss={st["mean_loss"]:.4f} | %>{ANOMALY_THRESHOLD}={st["pct_above_3"]:.2f}% | {st["elapsed_min"]:.1f} min')

    # AUTO-SAVE this sample to HuggingFace immediately
    print(f'  Saving {sid} to HuggingFace...', end=' ')
    save_sample_to_hf(sid)
    print('✓')

print(f'\nDone: {done} computed, {skipped} restored | {(time.time()-t_session)/3600:.1f} h')

In [ ]:
# Cell 9: Aggregate results with correct metadata merge
import pandas as pd, numpy as np, json

rows = [json.load(open(f)) for f in sorted(RESULTS_DIR.glob('*_anomaly.json'))]
stats_df = pd.DataFrame(rows).drop(columns=['hist_counts','hist_edges'], errors='ignore')

# Fix merge: FASTA sample_id is e.g. AAMO_R1, metadata is AAMO
stats_df['meta_key'] = stats_df['sample_id'].str.replace('_R1$', '', regex=True)
merged = stats_df.merge(meta_raw, left_on='meta_key', right_on='sample_id', how='left')
merged['sample_id'] = merged['sample_id_x']
merged = merged.drop(columns=['sample_id_x','sample_id_y','meta_key'], errors='ignore')
merged.to_csv(RESULTS_DIR / 'anomaly_summary.csv', index=False)

# Validate merge
n_nan = merged['city'].isna().sum()
print(f'Samples: {len(merged)} | NaN city: {n_nan}')
if n_nan > 0:
    print('WARNING: some samples did not merge — check sample IDs:')
    print('  Result IDs  :', stats_df['sample_id'].iloc[0])
    print('  Metadata IDs:', meta_raw['sample_id'].iloc[0])

# Build embedding matrix
emb_files  = sorted(RESULTS_DIR.glob('*_embedding.npy'))
sample_ids = [f.stem.replace('_embedding','') for f in emb_files]
emb_matrix = np.stack([np.load(f) for f in emb_files])
np.save(RESULTS_DIR/'embedding_matrix.npy', emb_matrix)
json.dump(sample_ids, open(RESULTS_DIR/'embedding_sample_ids.json','w'))

# Save summary to HF
HfApi().upload_file(
    path_or_fileobj=str(RESULTS_DIR/'anomaly_summary.csv'),
    path_in_repo='results/anomaly_summary.csv',
    repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
HfApi().upload_file(
    path_or_fileobj=str(RESULTS_DIR/'embedding_matrix.npy'),
    path_in_repo='results/embedding_matrix.npy',
    repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
HfApi().upload_file(
    path_or_fileobj=str(RESULTS_DIR/'embedding_sample_ids.json'),
    path_in_repo='results/embedding_sample_ids.json',
    repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)

print(f'Embedding matrix: {emb_matrix.shape}')
print(f'Mean loss: {merged["mean_loss"].mean():.4f} ± {merged["mean_loss"].std():.4f}')
print(f'Paper WW : 1.2400 ± 1.3100')
display(merged[['sample_id','city','season','mean_loss','pct_above_3']].head(8))

In [ ]:
# Cell 10: Anomaly score plots
import matplotlib.pyplot as plt, seaborn as sns, numpy as np, json, pandas as pd

stats_df = pd.read_csv(RESULTS_DIR/'anomaly_summary.csv')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0,0]
city_order = sorted(stats_df['city'].dropna().unique())
sns.boxplot(data=stats_df, x='city', y='mean_loss', order=city_order, palette='Set2', ax=ax)
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (~1.24)')
ax.axhline(5.22, color='orange', ls='--', lw=1.5, label='Paper human genome (~5.22)')
ax.axhline(ANOMALY_THRESHOLD, color='red', ls='--', lw=1.5, label=f'Threshold ({ANOMALY_THRESHOLD})')
ax.set_title('Mean Anomaly Loss by City', fontweight='bold'); ax.legend(fontsize=7)

ax = axes[0,1]
s_ord = [s for s in SEASON_ORDER if s in stats_df['season'].values]
sns.boxplot(data=stats_df, x='season', y='mean_loss', order=s_ord, palette='Set1', ax=ax)
ax.axhline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (~1.24)')
ax.axhline(5.22, color='orange', ls='--', lw=1.5, label='Paper human genome (~5.22)')
ax.set_title('Mean Anomaly Loss by Season', fontweight='bold'); ax.legend(fontsize=7)

ax = axes[1,0]
ss = stats_df.sort_values('pct_above_3', ascending=False)
ax.bar(range(len(ss)), ss['pct_above_3'],
       color=[CITY_COLORS.get(c,'grey') for c in ss['city'].fillna('Unknown')])
ax.set_xticks(range(len(ss)))
ax.set_xticklabels(ss['sample_id'], rotation=90, fontsize=5)
ax.set_title(f'% Reads > {ANOMALY_THRESHOLD} per Sample', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c,label=ct) for ct,c in CITY_COLORS.items()], fontsize=7)

ax = axes[1,1]
all_counts, edges = None, None
for sf in sorted(RESULTS_DIR.glob('*_anomaly.json')):
    d = json.load(open(sf))
    c = np.array(d['hist_counts'], dtype=float)
    all_counts = c if all_counts is None else all_counts+c
    edges = np.array(d['hist_edges'])
ctr = (edges[:-1]+edges[1:])/2
ax.bar(ctr, all_counts/all_counts.sum(), width=ctr[1]-ctr[0], color='steelblue', alpha=0.7, label='Gujarat WW')
ax.axvline(1.24, color='green', ls='--', lw=2, label='Paper WW (1.24)')
ax.axvline(5.22, color='orange', ls='--', lw=2, label='Human genome (5.22)')
ax.axvline(5.83, color='purple', ls='--', lw=2, label='Random DNA (5.83)')
ax.axvline(ANOMALY_THRESHOLD, color='red', ls='--', lw=1.5, label=f'Threshold ({ANOMALY_THRESHOLD})')
ax.set_title('Loss Distribution vs Paper References', fontweight='bold')
ax.set_xlabel('Length-normalised CE Loss'); ax.legend(fontsize=7)

plt.suptitle('METAGENE-1 Anomaly Scoring — Gujarat Wastewater Shotgun Metagenomics',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'anomaly_analysis.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'anomaly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved anomaly_analysis.pdf/.png')

In [ ]:
# Cell 11: PCA + UMAP with correct city/season colors
import numpy as np, json, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

emb_matrix = np.load(RESULTS_DIR/'embedding_matrix.npy')
with open(RESULTS_DIR/'embedding_sample_ids.json') as f: sample_ids = json.load(f)

# Build emb_df with correct metadata
emb_df = pd.DataFrame({'sample_id': sample_ids})
emb_df['meta_key'] = emb_df['sample_id'].str.replace('_R1$','',regex=True)
emb_df = emb_df.merge(meta_raw, left_on='meta_key', right_on='sample_id', how='left')
emb_df['sample_id'] = emb_df['sample_id_x']
emb_df = emb_df.drop(columns=['sample_id_x','sample_id_y','meta_key'], errors='ignore')

print(f'NaN city: {emb_df["city"].isna().sum()} / {len(emb_df)}')

norms    = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
emb_norm = emb_matrix / (norms+1e-8)

pca = PCA(n_components=50); pcs = pca.fit_transform(emb_norm)
var = pca.explained_variance_ratio_[:5]*100

try:
    import umap
    coords_umap = umap.UMAP(n_components=2,random_state=SEED,n_neighbors=10,min_dist=0.3).fit_transform(pcs[:,:20])
    do_umap = True
except: do_umap=False; print('UMAP skipped')

ncols = 3 if do_umap else 2
fig, axes = plt.subplots(1, ncols, figsize=(6*ncols, 5))

def scat(ax, x, y, title, xl, yl):
    idx = {s:i for i,s in enumerate(emb_df['sample_id'])}
    for _,row in emb_df.iterrows():
        i = idx[row['sample_id']]
        city   = row.get('city','') if pd.notna(row.get('city')) else 'Unknown'
        season = row.get('season','') if pd.notna(row.get('season')) else 'Unknown'
        ax.scatter(x[i], y[i],
                   c=CITY_COLORS.get(city,'grey'),
                   marker=SEASON_MARKERS.get(season,'o'),
                   s=80, edgecolors='white', linewidths=0.5, alpha=0.9)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel(xl); ax.set_ylabel(yl)

scat(axes[0],pcs[:,0],pcs[:,1],'PCA PC1 vs PC2',f'PC1 ({var[0]:.1f}%)',f'PC2 ({var[1]:.1f}%)')
scat(axes[1],pcs[:,0],pcs[:,2],'PCA PC1 vs PC3',f'PC1 ({var[0]:.1f}%)',f'PC3 ({var[2]:.1f}%)')
if do_umap: scat(axes[2],coords_umap[:,0],coords_umap[:,1],'UMAP','UMAP-1','UMAP-2')

ch = [mpatches.Patch(color=c,label=ct) for ct,c in CITY_COLORS.items()]
sh = [plt.scatter([],[],c='grey',marker=m,label=s,s=60) for s,m in SEASON_MARKERS.items()]
fig.legend(handles=ch+sh, ncol=4, loc='lower center', bbox_to_anchor=(0.5,-0.12), fontsize=8)
plt.suptitle('METAGENE-1 Sample Embeddings — Gujarat Wastewater', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'embedding_pca_umap.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'embedding_pca_umap.png', dpi=150, bbox_inches='tight')
plt.show()

city_labels = emb_df['city'].fillna('Unknown').values
season_labels = emb_df['season'].fillna('Unknown').values
if len(np.unique(city_labels))>1:
    print(f'Silhouette (city)  : {silhouette_score(emb_norm, city_labels):.4f}')
    print(f'Silhouette (season): {silhouette_score(emb_norm, season_labels):.4f}')
print(f'PCA var PC1-5: {np.round(var,2)}')
np.save(RESULTS_DIR/'pca_coords.npy', pcs)
if do_umap: np.save(RESULTS_DIR/'umap_coords.npy', coords_umap)

In [ ]:
# Cell 12: Pairwise cosine similarity heatmap
import numpy as np, json, matplotlib.pyplot as plt

emb_matrix = np.load(RESULTS_DIR/'embedding_matrix.npy')
emb_norm   = emb_matrix / (np.linalg.norm(emb_matrix,axis=1,keepdims=True)+1e-8)
sim        = emb_norm @ emb_norm.T

sr_map = {s:i for i,s in enumerate(SEASON_ORDER)}
edf    = emb_df.copy()
edf['_sr'] = edf['season'].map(sr_map).fillna(4)
edf    = edf.sort_values(['city','_sr'])
sidx   = [list(emb_df['sample_id']).index(s) for s in edf['sample_id']]
sim_s  = sim[np.ix_(sidx,sidx)]

fig, ax = plt.subplots(figsize=(13,11))
im = ax.imshow(sim_s, cmap='RdYlGn', vmin=0.9, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='Cosine similarity')
lbs = edf['sample_id'].values
ax.set_xticks(range(len(lbs))); ax.set_yticks(range(len(lbs)))
ax.set_xticklabels(lbs, rotation=90, fontsize=5)
ax.set_yticklabels(lbs, fontsize=5)
ax.set_title('Pairwise Cosine Similarity — Mean Embeddings (sorted city+season)', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'embedding_similarity_heatmap.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'embedding_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
np.save(RESULTS_DIR/'cosine_similarity_matrix.npy', sim)
print('Saved.')

In [ ]:
# Cell 13: Pathogen signal — season × city heatmap
import matplotlib.pyplot as plt, seaborn as sns, pandas as pd

stats_df = pd.read_csv(RESULTS_DIR/'anomaly_summary.csv')
fig, axes = plt.subplots(1, 2, figsize=(13,5))

ax = axes[0]
piv = stats_df.pivot_table(index='season',columns='city',values='pct_above_3',aggfunc='mean')
piv = piv.reindex([s for s in SEASON_ORDER if s in piv.index])
sns.heatmap(piv, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label':f'% reads > {ANOMALY_THRESHOLD}'})
ax.set_title('% Anomalous Reads by Season × City', fontweight='bold')

ax = axes[1]
for city,grp in stats_df.groupby('city'):
    ax.scatter(grp['mean_loss'], grp['pct_above_3'], label=city, s=70, alpha=0.8,
               c=CITY_COLORS.get(city,'grey'), edgecolors='white', linewidths=0.5)
ax.axvline(1.24, color='green', ls='--', lw=1.5, label='Paper WW (1.24)')
ax.axvline(5.22, color='orange', ls='--', lw=1.5, label='Human genome (5.22)')
ax.set_xlabel('Mean CE Loss'); ax.set_ylabel(f'% Reads > {ANOMALY_THRESHOLD}')
ax.set_title('Mean Loss vs % Anomalous Reads', fontweight='bold'); ax.legend(fontsize=8)

plt.suptitle('METAGENE-1 Pathogen Signal — Gujarat Wastewater', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'pathogen_signal.pdf', dpi=150, bbox_inches='tight')
plt.savefig(RESULTS_DIR/'pathogen_signal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 10 highest anomaly:')
print(stats_df.nlargest(10,'pct_above_3')[['sample_id','city','season','mean_loss','pct_above_3']].to_string(index=False))

In [ ]:
# Cell 14: Final summary — publication numbers
import pandas as pd, numpy as np
stats_df = pd.read_csv(RESULTS_DIR/'anomaly_summary.csv')

print('='*65)
print('METAGENE-1 — Gujarat Wastewater Shotgun Metagenomics')
print('='*65)
print(f'Samples: {len(stats_df)} | Reads/sample: {N_READS_PER_SAMPLE:,}')
print()
print('ANOMALY SCORES vs Liu et al. 2025')
print(f'  Gujarat WW mean CE loss   : {stats_df["mean_loss"].mean():.4f} ± {stats_df["mean_loss"].std():.4f}')
print(f'  Paper WW in-dist          : 1.2400 ± 1.3100')
print(f'  Paper human genome        : 5.2200 ± 0.2200')
print(f'  Paper random sequence     : 5.8300 ± 0.2900')
print(f'  % reads > 3.0 (mean)      : {stats_df["pct_above_3"].mean():.2f}%')
print()
print('BY CITY:')
print(stats_df.groupby('city')[['mean_loss','pct_above_3']].mean().round(4).to_string())
print()
print('BY SEASON:')
sp = [s for s in SEASON_ORDER if s in stats_df['season'].values]
print(stats_df.groupby('season')[['mean_loss','pct_above_3']].mean().reindex(sp).round(4).to_string())
print('='*65)

In [ ]:
# Cell 15: Save all plots and summaries to HuggingFace
from huggingface_hub import HfApi
api = HfApi()
print('Saving plots and summaries to HuggingFace...')
for ext in ['*.pdf','*.png','*.csv','*.npy','*.json']:
    for f in RESULTS_DIR.glob(ext):
        try:
            api.upload_file(
                path_or_fileobj=str(f),
                path_in_repo=f'results/{f.name}',
                repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
            print(f'  ✓ {f.name}')
        except Exception as e:
            print(f'  ✗ {f.name}: {e}')
print('All results permanently saved to HuggingFace.')